# Complete DEM Super-Resolution Data Pipeline

**End-to-end pipeline, no intermediate files between steps.**

| Cell | What it does |
|------|-------------|
| 1 | Config: all paths and hyperparameters |
| 2 | ICESat-2 KDTree spatial sieve |
| 3 | FABDEM 30 m to 10 m bicubic resample (in-memory MemoryFile) |
| 4 | HMA DSM: WGS84 to EGM2008 geoid + bare-earth DTM (WBT) + 8 m to 10 m downscale |
| 5 | 4-channel tensor assembly + quality gauntlet + .npz export |

**Inputs required:**
- Pre-compiled ICESat-2 CSV with columns: `longitude`, `latitude`, `height_orthometric`
- FABDEM 30 m GeoTIFF
- HMA 8 m DSM GeoTIFF
- EGM2008 geoid `.gtx` file

**Output:** `(4, H, W)` float32 tensors as `.npz` with embedded GDAL geotransform + EPSG code.

In [ ]:
# ============================================================
# CELL 1 -- CONFIGURATION
# Edit every path in this cell before running anything else.
# ============================================================
import os

# -- Input paths ---------------------------------------------
LIDAR_CSV_PATH  = r"D:\path\to\compiled_lidar.csv"    # pre-compiled ICESat-2 ATL08 CSV
FABDEM_30M_PATH = r"D:\path\to\fabdem_30m.tif"         # FABDEM 30 m source tile
HMA_DSM_8M_PATH = r"D:\path\to\hma_dsm_8m.tif"         # HMA 8 m DSM tile
GTX_FILE        = r"D:\path\to\egm08_25.gtx"            # EGM2008 geoid grid (.gtx)

# WhiteboxTools: folder containing whitebox_tools.exe
# Leave as None to auto-search under WBT_SEARCH_ROOT
WBT_EXE_DIR     = None
WBT_SEARCH_ROOT = r"D:\Vaibhav_00006"

# -- Output --------------------------------------------------
OUTPUT_DIR = r"D:\path\to\output_tensors"

# -- KDTree filter hyperparameters ---------------------------
KNN_NEIGHBORS       = 6
MAX_VERTICAL_DELTA  = 25.0   # metres: max elevation deviation from local median
MAX_HORIZONTAL_DIST = 50.0   # metres: max avg distance to nearest neighbours

# -- Patch / quality gauntlet hyperparameters ----------------
CROP_SIZE           = 256
PATCH_STRIDE        = 64      # sliding-window stride (= CROP_SIZE for non-overlapping)
MAX_HOLE_TOLERANCE  = 0.25    # max fraction of original NoData in a patch
MAX_ELEVATION_LIMIT = 5000.0  # snowline cutoff (metres)
MIN_STD_DEV         = 0.25    # min FABDEM std-dev (glacial lake / flat water guard)
MIN_PHOTONS         = 6       # minimum ICESat-2 photons per accepted patch
PHOTON_DEM_DELTA    = 10.0    # max abs diff (m) between photon z and GT DEM

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded. Edit paths above, then run cells in order.")

In [ ]:
# ============================================================
# CELL 2 -- ICESat-2 PHOTON KDTree SPATIAL SIEVE
# Input  : LIDAR_CSV_PATH  (from Cell 1)
# Output : filtered_xs, filtered_ys, filtered_zs  (numpy arrays in RAM)
# ============================================================
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import time, gc

print("Loading ICESat-2 photons from:", LIDAR_CSV_PATH)
t0 = time.time()

df        = pd.read_csv(LIDAR_CSV_PATH)
raw_count = len(df)
print(f"   -> {raw_count:,} raw photons loaded.")

xs_raw = df["longitude"].values.astype(np.float32)
ys_raw = df["latitude"].values.astype(np.float32)
zs_raw = df["height_orthometric"].values.astype(np.float32)
del df
gc.collect()

if raw_count > KNN_NEIGHBORS:
    print(f"   -> Building cKDTree (k={KNN_NEIGHBORS}) and running spatial sieve ...")

    coords_2d       = np.c_[xs_raw, ys_raw]
    tree            = cKDTree(coords_2d)
    distances, idxs = tree.query(coords_2d, k=KNN_NEIGHBORS)

    # Local statistics from k-1 true neighbours (col 0 = self, excluded)
    local_median = np.median(zs_raw[idxs[:, 1:]], axis=1)
    avg_dist     = np.mean(distances[:, 1:], axis=1)

    # Photon survives if it fits local terrain OR is too isolated to be judged
    vertical_ok   = np.abs(zs_raw - local_median) < MAX_VERTICAL_DELTA
    is_isolated   = avg_dist >= MAX_HORIZONTAL_DIST
    survival_mask = vertical_ok | is_isolated

    filtered_xs = xs_raw[survival_mask].copy()
    filtered_ys = ys_raw[survival_mask].copy()
    filtered_zs = zs_raw[survival_mask].copy()

    dropped = raw_count - len(filtered_zs)
    pct     = dropped / raw_count * 100
    print(f"   -> Removed {dropped:,} spatial anomalies ({pct:.2f}%).")

    del tree, coords_2d, distances, idxs, local_median, avg_dist
    del xs_raw, ys_raw, zs_raw
    gc.collect()
else:
    print("   WARNING: Too few points for KDTree -- skipping spatial sieve.")
    filtered_xs, filtered_ys, filtered_zs = xs_raw, ys_raw, zs_raw

print(f"\nRetained {len(filtered_zs):,} clean photons in {time.time() - t0:.1f}s.")
print(f"   Longitude range : {filtered_xs.min():.4f} -> {filtered_xs.max():.4f}")
print(f"   Latitude  range : {filtered_ys.min():.4f} -> {filtered_ys.max():.4f}")
print(f"   Elevation range : {filtered_zs.min():.1f} m -> {filtered_zs.max():.1f} m")

In [ ]:
# ============================================================
# CELL 3 -- FABDEM 30 m -> 10 m BICUBIC RESAMPLE  (in-memory)
# Input  : FABDEM_30M_PATH  (from Cell 1)
# Output : fab_memfile  (rasterio MemoryFile, kept open for Cell 5)
# ============================================================
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.io import MemoryFile
import gc

print("Loading FABDEM from:", FABDEM_30M_PATH)

with rasterio.open(FABDEM_30M_PATH) as src:
    orig_h, orig_w  = src.height, src.width
    orig_res_x, _   = src.res

    # 30 m -> 10 m: 3x more pixels per side
    FABDEM_SCALE = 3.0
    new_h = int(orig_h * FABDEM_SCALE)
    new_w = int(orig_w * FABDEM_SCALE)

    print(f"   Original : {orig_w} x {orig_h} px  @  {orig_res_x:.1f} m/px")
    print(f"   Target   : {new_w} x {new_h} px  @  {orig_res_x / FABDEM_SCALE:.1f} m/px")
    print("   Resampling with bicubic (cubic) interpolation ...")

    data = src.read(
        out_shape=(src.count, new_h, new_w),
        resampling=Resampling.cubic,
    )

    # Pixel size shrinks by factor 3, so scale the affine transform accordingly
    new_transform = src.transform * Affine.scale(1.0 / FABDEM_SCALE, 1.0 / FABDEM_SCALE)

    fab_meta = src.meta.copy()
    fab_meta.update({"height": new_h, "width": new_w, "transform": new_transform})

    # Store in MemoryFile so WarpedVRT can open it in Cell 5 without disk I/O
    fab_memfile = MemoryFile()
    with fab_memfile.open(**fab_meta) as mem_dst:
        mem_dst.write(data)

del data
gc.collect()

print("\nFABDEM 10 m held in memory.")
print(f"   CRS   : {fab_meta['crs']}")
print(f"   Dtype : {fab_meta['dtype']}")
print(f"   Shape : {new_w} x {new_h}")

In [ ]:
# ============================================================
# CELL 4 -- HMA DSM PIPELINE
#   Step A : WGS84 Ellipsoid -> EGM2008 Orthometric  (in-memory)
#   Step B : 3-pass progressive bare-earth DTM        (WBT, temp disk, auto-deleted)
#   Step C : 8 m -> 10 m fractional-consensus downscale (in-memory)
#
# Input  : HMA_DSM_8M_PATH, GTX_FILE, WBT_EXE_DIR / WBT_SEARCH_ROOT
# Output : hma_dtm_arr, hma_transform, hma_crs, hma_bounds, hma_meta
# ============================================================
import os, gc, shutil, tempfile
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from pathlib import Path
import whitebox

# -- Locate WhiteboxTools -------------------------------------
whitebox.whitebox_tools.download_wbt = lambda *args, **kwargs: None  # stay offline
wbt = whitebox.WhiteboxTools()

if WBT_EXE_DIR:
    wbt.set_whitebox_dir(WBT_EXE_DIR)
    print("   -> WhiteboxTools dir (manual):", WBT_EXE_DIR)
else:
    exe_list = list(Path(WBT_SEARCH_ROOT).rglob("whitebox_tools.exe"))
    if not exe_list:
        raise FileNotFoundError(
            f"whitebox_tools.exe not found under {WBT_SEARCH_ROOT}.\n"
            "Set WBT_EXE_DIR in Cell 1 or extract the WBT zip file."
        )
    wbt.set_whitebox_dir(str(exe_list[0].parent))
    print("   -> WhiteboxTools auto-found:", exe_list[0].parent)
wbt.set_verbose_mode(False)

# -- Step A : Geoid correction (WGS84 -> EGM2008) ------------
print(f"\n[A] Applying EGM2008 geoid shift to {Path(HMA_DSM_8M_PATH).name} ...")

with rasterio.open(HMA_DSM_8M_PATH) as dsm:
    dsm_arr    = dsm.read(1).astype(np.float32)
    dsm_meta   = dsm.meta.copy()
    dsm_nodata = float(dsm.nodata) if dsm.nodata is not None else -9999.0

    with rasterio.open(GTX_FILE) as gtx:
        geoid_grid = np.zeros(dsm_arr.shape, dtype=np.float32)
        gtx_crs    = gtx.crs if gtx.crs is not None else "EPSG:4326"
        # Reproject geoid undulation surface onto DSM pixel grid
        reproject(
            source=rasterio.band(gtx, 1),
            destination=geoid_grid,
            src_transform=gtx.transform,
            src_crs=gtx_crs,
            dst_transform=dsm.transform,
            dst_crs=dsm.crs,
            resampling=Resampling.cubic,
        )

    valid_mask = (dsm_arr != dsm_nodata) & (~np.isnan(dsm_arr))
    # Orthometric height = Ellipsoidal height - Geoid undulation (N value)
    ortho_arr  = np.where(valid_mask, dsm_arr - geoid_grid, dsm_nodata).astype(np.float32)
    dsm_meta.update(dtype="float32", nodata=dsm_nodata)

del geoid_grid, dsm_arr
gc.collect()
print(f"   -> Valid pixels: {int(valid_mask.sum()):,} / {valid_mask.size:,}")

# -- Step B : Bare-earth DTM via WhiteboxTools (temp disk, auto-deleted) -------
print("\n[B] Running 3-pass progressive bare-earth extraction (WhiteboxTools) ...")

_tmpdir = tempfile.mkdtemp(prefix="hma_pipeline_")
try:
    _ortho_tif = os.path.join(_tmpdir, "ortho_8m.tif")
    _pass1     = os.path.join(_tmpdir, "pass1.tif")
    _pass2     = os.path.join(_tmpdir, "pass2.tif")
    _dtm8_tif  = os.path.join(_tmpdir, "dtm_8m.tif")

    # Write geoid-corrected DSM to temp file (WBT requires a file path)
    with rasterio.open(_ortho_tif, "w", **dsm_meta) as tmp_out:
        tmp_out.write(ortho_arr, 1)
    del ortho_arr
    gc.collect()

    print("   -> Pass 1/3  kernel=6,  slope=35 deg  (short structures / low canopy)")
    wbt.remove_off_terrain_objects(_ortho_tif, _pass1,    filter=6,  slope=35.0)

    print("   -> Pass 2/3  kernel=16, slope=45 deg  (taller vegetation)")
    wbt.remove_off_terrain_objects(_pass1,     _pass2,    filter=16, slope=45.0)

    print("   -> Pass 3/3  kernel=36, slope=55 deg  (buildings / ridge artefacts)")
    wbt.remove_off_terrain_objects(_pass2,     _dtm8_tif, filter=36, slope=55.0)

    # -- Step C : 8 m -> 10 m fractional-void-consensus downscale (in-memory) -
    print("\n[C] Downscaling DTM 8 m -> 10 m with void-aware fractional consensus ...")

    with rasterio.open(_dtm8_tif) as dtm8:
        dtm8_arr       = dtm8.read(1).astype(np.float32)
        nd             = float(dtm8.nodata) if dtm8.nodata is not None else -9999.0
        b              = dtm8.bounds
        dtm8_transform = dtm8.transform
        dtm8_crs       = dtm8.crs

        # Compute output grid dimensions at exactly 10 m
        dst_w = int(round((b.right - b.left)   / 10.0))
        dst_h = int(round((b.top   - b.bottom)  / 10.0))
        dst_transform = rasterio.transform.from_bounds(
            b.left, b.bottom, b.right, b.top, dst_w, dst_h
        )

        # Void-fraction layer: 1.0 = no-data, 0.0 = valid
        is_void = ((dtm8_arr == nd) | np.isnan(dtm8_arr)).astype(np.float32)

        dst_elev = np.zeros((dst_h, dst_w), dtype=np.float32)
        reproject(
            source=rasterio.band(dtm8, 1), destination=dst_elev,
            src_transform=dtm8_transform, src_crs=dtm8_crs,
            dst_transform=dst_transform,  dst_crs=dtm8_crs,
            resampling=Resampling.bilinear,
        )

        dst_void_frac = np.zeros((dst_h, dst_w), dtype=np.float32)
        reproject(
            source=is_void, destination=dst_void_frac,
            src_transform=dtm8_transform, src_crs=dtm8_crs,
            dst_transform=dst_transform,  dst_crs=dtm8_crs,
            resampling=Resampling.average,
        )

        # Mark a 10 m output pixel void if >50% of contributing 8 m cells were void
        hma_dtm_arr = np.where(dst_void_frac >= 0.5, nd, dst_elev).astype(np.float32)

        hma_meta = dtm8.meta.copy()
        hma_meta.update(
            transform=dst_transform, width=dst_w, height=dst_h,
            nodata=nd, dtype="float32",
        )
        hma_transform = dst_transform
        hma_crs       = dtm8_crs
        hma_bounds    = b   # geographic extent unchanged by resolution change

    del dtm8_arr, is_void, dst_elev, dst_void_frac

finally:
    shutil.rmtree(_tmpdir, ignore_errors=True)
    gc.collect()

print("\nHMA DTM 10 m held in memory.")
print(f"   Shape      : {hma_dtm_arr.shape}  (H x W)")
print(f"   CRS        : {hma_crs}")
print(f"   Resolution : 10 m x 10 m")
print(f"   Void px    : {int((hma_dtm_arr == nd).sum()):,}")

In [ ]:
# ============================================================
# CELL 5 -- 4-CHANNEL TENSOR ASSEMBLY + QUALITY GAUNTLET + EXPORT
#
# Channel layout saved in each .npz:
#   Ch 0  -- FABDEM 10 m (coarse input, warped onto HMA grid)
#   Ch 1  -- ICESat-2 bare-earth truth (photons gridded, min-z per pixel)
#   Ch 2  -- ICESat-2 photon occupancy mask (1=has photon, 0=empty)
#   Ch 3  -- HMA DTM 10 m EGM2008 (ground truth target, holes filled)
#
# Each .npz also stores:
#   geotransform  -- GDAL 6-float: (x0, dx, 0, y0, 0, -dy)
#   epsg          -- integer EPSG code of the HMA tile CRS
#
# Quality gauntlet (fail-fast, cheapest checks first):
#   1. GT original-NoData hole fraction  > MAX_HOLE_TOLERANCE   -> reject
#   2. Mean FABDEM elevation             > MAX_ELEVATION_LIMIT  -> reject (snowline)
#   3. FABDEM std dev                    < MIN_STD_DEV          -> reject (glacial lake)
#   4. >60% gradient pixels with slope   < 0.05 px/px           -> reject (flat/ramp)
#   5. Photon count                      < MIN_PHOTONS          -> reject (too sparse)
# ============================================================
import os, gc
import numpy as np
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.fill import fillnodata
from rasterio.transform import rowcol
from rasterio.warp import transform_bounds
from pyproj import Transformer
import time

t_start = time.time()
h, w    = hma_dtm_arr.shape
nd      = float(hma_meta["nodata"])

# -- 1. Build GT validity mask and fill voids -----------------
print("[1/4] Preparing HMA DTM ground-truth layer ...")

ch4_gt_mask = ((hma_dtm_arr != nd) & (~np.isnan(hma_dtm_arr))).astype(np.uint8)
ch4_gt_dem  = hma_dtm_arr.copy()
ch4_gt_dem[ch4_gt_mask == 0] = 0.0

missing_gt = int(ch4_gt_mask.size - ch4_gt_mask.sum())
if missing_gt > 0:
    # Fill voids so the model gets a complete surface.
    # The validity mask (ch4_gt_mask) tells the model where real data is.
    ch4_gt_dem = fillnodata(
        ch4_gt_dem, mask=ch4_gt_mask,
        max_search_distance=250, smoothing_iterations=1,
    )
print(f"   -> GT shape: {ch4_gt_dem.shape}  |  Holes filled: {missing_gt:,} px")

# -- 2. Warp in-memory FABDEM 10 m onto HMA grid --------------
print("\n[2/4] Warping FABDEM 10 m onto HMA grid (bilinear WarpedVRT) ...")

with fab_memfile.open() as fab_src:
    vrt_opts = {
        "crs":        hma_crs,
        "transform":  hma_transform,
        "height":     h,
        "width":      w,
        "resampling": Resampling.bilinear,
    }
    with WarpedVRT(fab_src, **vrt_opts) as vrt:
        ch1_fabdem = vrt.read(1).astype(np.float32)
        fab_nd     = vrt.nodata

fab_vm = (
    (ch1_fabdem != fab_nd) & (~np.isnan(ch1_fabdem))
).astype(np.uint8) if fab_nd is not None else (~np.isnan(ch1_fabdem)).astype(np.uint8)

ch1_fabdem[fab_vm == 0] = 0.0
if int(fab_vm.size - fab_vm.sum()) > 0:
    ch1_fabdem = fillnodata(
        ch1_fabdem, mask=fab_vm, max_search_distance=100, smoothing_iterations=1,
    )
print(f"   -> FABDEM aligned. Shape: {ch1_fabdem.shape}")

# -- 3. Grid ICESat-2 photons onto HMA raster -----------------
print("\n[3/4] Gridding ICESat-2 photons onto HMA raster ...")

hma_bb  = transform_bounds(hma_crs, "EPSG:4326", *hma_bounds)
bb_mask = (
    (filtered_xs >= hma_bb[0]) & (filtered_xs <= hma_bb[2]) &
    (filtered_ys >= hma_bb[1]) & (filtered_ys <= hma_bb[3])
)
tile_xs_geo = filtered_xs[bb_mask]
tile_ys_geo = filtered_ys[bb_mask]
tile_zs     = filtered_zs[bb_mask]
print(f"   -> {len(tile_zs):,} photons within tile bounding box.")

ch2_truth = np.zeros((h, w), dtype=np.float32)
ch3_mask  = np.zeros((h, w), dtype=np.float32)

if len(tile_zs) > 0:
    # Reproject lon/lat -> HMA projected CRS
    transformer = Transformer.from_crs("EPSG:4326", hma_crs, always_xy=True)
    tile_xs_proj, tile_ys_proj = transformer.transform(tile_xs_geo, tile_ys_geo)

    rows_px, cols_px = rowcol(hma_transform, tile_xs_proj, tile_ys_proj)
    rows_px = np.array(rows_px, dtype=np.int64)
    cols_px = np.array(cols_px, dtype=np.int64)

    # Filter A: within raster extent
    in_raster = (rows_px >= 0) & (rows_px < h) & (cols_px >= 0) & (cols_px < w)
    rows_px, cols_px, tile_zs = rows_px[in_raster], cols_px[in_raster], tile_zs[in_raster]

    # Filter B: must land on an original valid GT pixel (not a filled void)
    on_valid = ch4_gt_mask[rows_px, cols_px] == 1
    rows_px, cols_px, tile_zs = rows_px[on_valid], cols_px[on_valid], tile_zs[on_valid]

    # Filter C: photon elevation must agree with GT DEM within +/- PHOTON_DEM_DELTA
    dem_agree = np.abs(tile_zs - ch4_gt_dem[rows_px, cols_px]) < PHOTON_DEM_DELTA
    rows_px, cols_px, tile_zs = rows_px[dem_agree], cols_px[dem_agree], tile_zs[dem_agree]

    # Grid: minimum elevation per pixel (bare-earth / ground selection)
    for r, c, z in zip(rows_px, cols_px, tile_zs):
        if ch3_mask[r, c] == 1.0:
            ch2_truth[r, c] = min(ch2_truth[r, c], z)
        else:
            ch2_truth[r, c] = z
            ch3_mask[r, c]  = 1.0

    print(f"   -> Photons after all filters : {len(tile_zs):,}")
    print(f"   -> Occupied raster pixels    : {int(ch3_mask.sum()):,}")
else:
    print("   WARNING: No photons fall within this tile.")

# Stack 4-channel full-tile tensor
full_tensor = np.stack([ch1_fabdem, ch2_truth, ch3_mask, ch4_gt_dem], axis=0)
print(f"\n   Full tensor: {full_tensor.shape}  (4 x H x W)")

# -- 4. Sliding-window patch extraction + quality gauntlet + save -------------
print("\n[4/4] Sliding-window patch extraction with quality gauntlet ...")


def _axis_offsets(total, crop, stride):
    """Sliding-window start offsets; always includes a flush-edge window."""
    if total < crop:  return []
    if total == crop: return [0]
    offs = list(range(0, total - crop + 1, stride))
    if offs[-1] != total - crop:
        offs.append(total - crop)
    return offs


y_offs = _axis_offsets(h, CROP_SIZE, PATCH_STRIDE)
x_offs = _axis_offsets(w, CROP_SIZE, PATCH_STRIDE)
print(f"   -> {len(y_offs) * len(x_offs):,} candidate windows ({len(y_offs)} y x {len(x_offs)} x)")

try:
    epsg_code = int(hma_crs.to_epsg() or 0)
except Exception:
    epsg_code = 0

saved = 0
rej   = {"holes": 0, "snow": 0, "lake": 0, "flat": 0, "photon": 0}

for y in y_offs:
    for x in x_offs:

        gt_mask_patch = ch4_gt_mask[y:y + CROP_SIZE, x:x + CROP_SIZE]

        # GAUNTLET 1 -- original-NoData hole fraction
        hole_frac = 1.0 - float(gt_mask_patch.sum()) / float(gt_mask_patch.size)
        if hole_frac > MAX_HOLE_TOLERANCE:
            rej["holes"] += 1
            continue

        patch     = full_tensor[:, y:y + CROP_SIZE, x:x + CROP_SIZE]
        fab_patch = patch[0]

        # GAUNTLET 2 -- above permanent snowline
        if np.mean(fab_patch) > MAX_ELEVATION_LIMIT:
            rej["snow"] += 1
            continue

        # GAUNTLET 3 -- glacial lake / flat water (std-dev guard)
        if np.std(fab_patch) < MIN_STD_DEV:
            rej["lake"] += 1
            continue

        # GAUNTLET 4 -- featureless flat plain / tilted ramp
        dy_g, dx_g = np.gradient(fab_patch)
        slope_mag  = np.sqrt(dx_g ** 2 + dy_g ** 2)
        if np.sum(slope_mag < 0.05) / slope_mag.size > 0.60:
            rej["flat"] += 1
            continue

        # GAUNTLET 5 -- minimum photon occupancy
        photon_count = int(patch[2].sum())
        if photon_count < MIN_PHOTONS:
            rej["photon"] += 1
            continue

        # Passed all gates -- compute patch geotransform and save
        # GDAL convention: (x_topleft, pixel_w, row_rot, y_topleft, col_rot, -pixel_h)
        patch_gt = np.array([
            hma_transform.c + x * hma_transform.a,   # x0 of patch top-left corner
            hma_transform.a,                           # dx  (e.g. +10.0)
            0.0,                                       # row rotation (0 for north-up)
            hma_transform.f + y * hma_transform.e,   # y0 of patch top-left corner
            0.0,                                       # col rotation (0 for north-up)
            hma_transform.e,                           # -dy (e.g. -10.0)
        ], dtype=np.float64)

        fname = os.path.join(OUTPUT_DIR, f"{photon_count:05d}_p{saved:05d}.npz")
        np.savez_compressed(
            fname,
            tensor=patch.astype(np.float32),   # (4, CROP_SIZE, CROP_SIZE)
            geotransform=patch_gt,              # (6,)  use to write prediction GeoTIFF
            epsg=np.int32(epsg_code),
        )
        saved += 1

elapsed = time.time() - t_start
print("\n" + "=" * 60)
print("              PIPELINE COMPLETE")
print("=" * 60)
print(f"  Tensors saved              : {saved:,}")
print(f"  Rejected [Holes]           : {rej['holes']:,}")
print(f"  Rejected [Snowline]        : {rej['snow']:,}")
print(f"  Rejected [Glacial Lake]    : {rej['lake']:,}")
print(f"  Rejected [Flat/Ramp]       : {rej['flat']:,}")
print(f"  Rejected [Low Photon]      : {rej['photon']:,}")
print(f"  Total pipeline time        : {elapsed:.1f}s")
print(f"  Output dir                 : {OUTPUT_DIR}")
print("\nTo load a saved tensor:")
print("  z      = np.load('00042_p00000.npz')")
print("  tensor = z['tensor']        # shape (4, H, W) float32")
print("  gt6    = z['geotransform']  # (x0, dx, 0, y0, 0, -dy)")
print("  epsg   = int(z['epsg'])     # e.g. 32644")